# 第87课 · 把 7B 模型塞进笔记本——INT8 量化（quantization）原理与本地推理

**学习目标**
- 理解 float32 → int8 量化的原理（scale + zero_point）
- 从零实现 INT8 量化/反量化（Dequantization）——注意：此处演示无符号 **uint8**（0–255）；工业 INT8 为有符号（-128..127）
- 测量量化误差（Quantization Error） vs 比特数
- （可选）用 HuggingFace 连接真实本地 LLM

← **上一课**　[L86 · 采样策略从零实现](L86_sampling.ipynb)

> 上节课学习了 **采样策略从零实现**：temperature / top-k / top-p，纯 NumPy。  
> 本课将探讨 **量化与本地推理**。

## 有没有想过：7B 模型为什么能跑在你的笔记本上？

几年前，运行一个 7B 参数的语言模型需要一块 A100（80 GB VRAM，价格约 $10,000）。
今天，同一个模型可以在 MacBook M2 的统一内存里流畅运行——只需要**量化（Quantization）**。

**直觉**：float32 精确但"胖"——每个数占 4 字节。int8 不精确但"瘦"——每个数只占 1 字节。
把模型权重从 float32 压缩成 int8，内存直接降到 1/4。

**核心问题**：如何用一个整数"代表"一段浮点数区间？

**答案是仿射量化（Affine Quantization）**：

```
scale     = (x_max - x_min) / 255   （每格代表多大的 float 步长）
zero_point = round(-x_min / scale)   （把0.0映射到哪个整数格）
x_q       = clip(round(x/scale) + zero_point, 0, 255)  → uint8
x_deq     = (x_q - zero_point) × scale                 → float32（近似还原）
```

**量化误差**：每个数最多损失 scale/2 的精度——8位时平均误差约 ≈ range/512，非常小。
当你减小到4位（16格），误差增大 16倍，这就是为什么 INT4 量化需要特殊校正。

```
float32: 7B × 4B = 28 GB → 需要 A100
int8:    7B × 1B =  7 GB → RTX 3080 够了
int4:    7B × 0.5B = 3.5 GB → 普通游戏 GPU 也行
```

本节演示 uint8 仿射量化的完整流程，并练习实现 `batch_quantize_error` 分析函数。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def quantize_int8(x: np.ndarray):
    """将 float32 权重量化为 uint8 (0..255，非对称量化)，返回 (quantized, scale, zero_point)。
    注意：返回值类型是 uint8（无符号，范围 0-255），不是有符号 int8（-128..127）。"""
    x_min, x_max = x.min(), x.max()
    scale = (x_max - x_min) / 255.0  # uint8 范围 0-255 (unsigned)
    if scale < 1e-8:
        return np.zeros_like(x, dtype=np.uint8), 1.0, 0
    zero_point = int(np.clip(round(-x_min / scale), 0, 255))
    x_q = np.round(x / scale).astype(np.int32) + zero_point
    x_q = np.clip(x_q, 0, 255).astype(np.uint8)
    return x_q, scale, zero_point

def dequantize_int8(x_q: np.ndarray, scale: float, zero_point: int) -> np.ndarray:
    """反量化：uint8 → float32。"""
    return (x_q.astype(np.float32) - zero_point) * scale

# 测试
np.random.seed(0)
W = np.random.randn(64, 64).astype(np.float32)   # 模拟权重矩阵

try:
    W_q, scale, zp = quantize_int8(W)
    assert W_q is not None, "⬜ 未实现"
    assert W_q.dtype == np.uint8, f"量化结果应为 uint8，得到 {W_q.dtype}"
    assert W_q.shape == W.shape, f"量化结果形状应与输入一致，得到 {W_q.shape}"

    W_deq = dequantize_int8(W_q, scale, zp)
    error = np.abs(W - W_deq)

    print(f'原始范围: [{W.min():.3f}, {W.max():.3f}]')
    print(f'Scale: {scale:.6f}, Zero point: {zp}')
    print(f'量化误差: mean={error.mean():.4f}, max={error.max():.4f}')
    print(f'内存: float32={W.nbytes} bytes → uint8={W_q.nbytes} bytes  ({W.nbytes/W_q.nbytes:.1f}× 压缩)')

    assert error.mean() < 0.05, f"量化误差过大: {error.mean():.4f}"
    print(f'✅ 量化验证通过，平均误差={error.mean():.4f}')
except (NotImplementedError, TypeError):
    print('⬜ 未实现')

> **⚠ 数据类型说明**：本课演示使用 `np.uint8`（无符号整数，范围 0–255）实现仿射量化。
> 工业界的 INT8 通常指**有符号** `int8`（范围 -128–127），使用对称量化（zero_point=0）。
> 两者精度相同（8 位），区别在于：
> - `uint8`：适合权重值域偏正的场景（如 ReLU 后激活值 ≥ 0）
> - `int8`：适合权重值域对称于零的场景（如线性层权重）
>
> 本课代码可直接改为 `np.int8` + 对称量化，留作课后练习。

In [ ]:
# 误差 vs 比特数
print(f'{"位宽":>4}  {"levels":>7}  {"均值误差":>10}  {"最大误差":>10}')
print('-' * 40)
for bits in [8, 6, 4, 3, 2]:
    levels = 2**bits - 1
    scale_b = (W.max() - W.min()) / levels
    W_q_b = np.clip(np.round((W - W.min()) / scale_b), 0, levels)
    W_r = W_q_b * scale_b + W.min()  # 等价写法：直接减 x_min，无显式 zero_point
    err = np.abs(W - W_r)
    print(f'{bits:>4}  {levels:>7}  {err.mean():>10.4f}  {err.max():>10.4f}')

## ✏️ 练习：`batch_quantize_error(W, bits_list)`

量化分析工程师的日常任务：对比同一权重矩阵在不同位宽下的量化误差，决定哪个 bit-width 在精度和内存之间取得最佳平衡。

**签名**：
```python
def batch_quantize_error(W: np.ndarray, bits_list: list) -> dict:
    # 返回 {bits: {"scale": float, "mean_err": float, "max_err": float}}
```

**3步实现路线**：

| 步骤 | 操作 | 代码提示 |
|---|---|---|
| 1 | 对每个 bits 计算 scale | `scale = (W.max()-W.min()) / (2**bits - 1)` |
| 2 | 量化 + 反量化 | `W_q = clip(round((W-W.min())/scale), 0, 2**bits-1)` → `W_r = W_q*scale + W.min()` |
| 3 | 计算并返回误差统计 | `{"scale": scale, "mean_err": abs_err.mean(), "max_err": abs_err.max()}` |

**验收标准**：
- `bits=8` 的 `mean_err` < 0.05（8位误差小）
- `bits=2` 的 `mean_err` > `bits=8` 的 `mean_err`（更少位宽误差更大）
- 返回值是 dict，键为 bits 整数

In [ ]:
def batch_quantize_error(W: np.ndarray, bits_list: list) -> dict:
    """对比不同位宽的量化误差。
    
    Args:
        W: float32 权重矩阵
        bits_list: 位宽列表，如 [8, 6, 4, 3, 2]
    
    Returns:
        {bits: {"scale": float, "mean_err": float, "max_err": float}}
    """
    # TODO: 遍历 bits_list，对每个 bits 计算仿射量化误差
    raise NotImplementedError("请实现 batch_quantize_error")


# ── 验证 ──────────────────────────────────────────────────────────────────────
np.random.seed(0)
W_test = np.random.randn(64, 64).astype(np.float32)

try:
    result = batch_quantize_error(W_test, [8, 4, 2])
    assert isinstance(result, dict), f"应返回 dict，得到 {type(result)}"
    assert 8 in result and 2 in result, f"结果缺少键，得到 {list(result.keys())}"
    assert result[2]["mean_err"] > result[8]["mean_err"], \
        f"位宽越小误差越大：bits=2 mean_err={result[2]['mean_err']:.4f} 应 > bits=8 {result[8]['mean_err']:.4f}"
    assert result[8]["mean_err"] < 0.05, f"8位误差应<0.05，得到 {result[8]['mean_err']:.4f}"
    print("✅ batch_quantize_error 验证通过：")
    for bits, stats in sorted(result.items()):
        print(f"  bits={bits}: scale={stats['scale']:.4f}, mean_err={stats['mean_err']:.4f}, max_err={stats['max_err']:.4f}")
except (NotImplementedError, TypeError):
    print("⚠️ 尚未实现 batch_quantize_error")

## ✏️ 闭卷推导检查格 — 量化数学

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：权重矩阵范围 `[-2.0, 2.0]`，使用 uint8（0–255）仿射量化。

1. 计算 `scale`（精确到小数点后4位）
2. 计算 `zero_point`（整数，使 0.0 精确映射）
3. 量化 `x = 0.5` 的结果 `x_q` 是多少？
4. 反量化 `x_q = 200` 得到的浮点值是多少？
5. 量化误差的理论上界是？（用 scale 表示）

（在此处写推导...）

In [ ]:
# ✏️ 本课自评
l87_review = {
    "quantization_intuition":    None,  # 理解量化=用整数格代表浮点区间，scale=range/levels？True/False
    "affine_formula":            None,  # 能写出scale/zero_point/量化/反量化完整公式？True/False
    "batch_error_impl":          None,  # batch_quantize_error()实现正确，验证通过？True/False
    "bit_tradeoff":              None,  # 理解位宽↓→scale↑→误差↑，4×内存压缩来自float32→int8？True/False
    "whiteboard_passed":         None,  # 白板挑战量化数学推导通关？True/False
}

unfilled = [k for k, v in l87_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l87_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L87 全部通关！进入 L88：TF-IDF 检索从零实现')

In [ ]:
# 验证：手算白板题
import numpy as np

x_min, x_max = -2.0, 2.0
bits = 8
levels = 2**bits - 1  # 255

scale_ans = (x_max - x_min) / levels
zp_ans = int(round(-x_min / scale_ans))
x_q_ans = int(np.clip(round(0.5 / scale_ans) + zp_ans, 0, 255))
x_deq_ans = (200 - zp_ans) * scale_ans
max_error_ans = scale_ans / 2.0

print(f"1. scale = ({x_max}-({x_min}))/{levels} = {scale_ans:.4f}")
print(f"2. zero_point = round(-({x_min})/{scale_ans:.4f}) = {zp_ans}")
print(f"3. x_q for x=0.5: round(0.5/{scale_ans:.4f})+{zp_ans} = {x_q_ans}")
print(f"4. x_deq for x_q=200: (200-{zp_ans})×{scale_ans:.4f} = {x_deq_ans:.4f}")
print(f"5. 量化误差上界 = scale/2 = {max_error_ans:.4f}")

assert abs(scale_ans - 4.0/255) < 1e-9
assert zp_ans == 128
assert abs(x_deq_ans - (200 - 128) * 4/255) < 1e-6
print("✅ 量化数学推导验证通过")

## 可选：连接 HuggingFace 本地推理

> 以下代码需要安装 `transformers` 和适当的模型权重。
> 如果跳过，前面的量化原理部分已经完整展示核心算法。

⚠️ **注意**：`model.generate()` 是 HuggingFace 的封装，这里作为工程集成使用，核心推理算法（KV-Cache、采样）见 L85-L86。

In [ ]:
# 工程集成示例（需要 transformers）
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    # 使用小型模型演示（替换为你有的本地模型）
    MODEL_ID = 'Qwen/Qwen2.5-0.5B'   # ~1GB，可本地运行

    print(f'Loading {MODEL_ID}...')
    # 实际运行时取消注释：
    # tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    # model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
    # inputs = tokenizer('The Fourier transform', return_tensors='pt')
    # with torch.no_grad():
    #     out = model.generate(**inputs, max_new_tokens=30, temperature=0.8, do_sample=True)
    # print(tokenizer.decode(out[0], skip_special_tokens=True))
    print('（已注释 — 取消注释并有对应模型权重后可运行）')
    print('核心算法（KV-Cache、采样）见 L85 和 L86 的从零实现')

except ImportError:
    print('transformers 未安装 — 跳过工程集成部分')
    print('前面的量化原理（INT8 quantize/dequantize）已完整运行')

## 本课收束

| 概念 | 要记住的 |
|---|---|
| INT8 量化 | scale = range/255, error ≈ scale/2 |
| 4× 内存压缩 | float32 → int8：显存（Video RAM，VRAM）需求降到 1/4 |
| 权衡 | 更少比特 → 更小模型 → 更大量化误差 |
| 工程集成 | HuggingFace = 便捷封装，不替代从零理解 |
| L88 | RAG 检索——不需要大模型也能做智能问答 |

下一课（L88）实现 TF-IDF 稀疏检索，把查询词映射到文档空间，这是 RAG 管道的第一个组件。

---

→ **下一课**　[L88 · TF-IDF 检索从零实现](L88_tfidf_retrieval.ipynb)

> 下节课将学习 **TF-IDF 检索从零实现**：词频逆文档频率，纯 NumPy 向量检索。